In [20]:
import numpy as np
import pandas as pd

In [21]:
#Load Dataset
import pandas as pd
df_movies = pd.read_csv(r'E:\Pyhton code\Data Sets\archive\ml-latest-small\ml-latest-small\movies.csv')
df_movies.head()

df_ratings = pd.read_csv(r'E:\Pyhton code\Data Sets\archive\ml-latest-small\ml-latest-small\ratings.csv')
df_ratings.head()


,userId,movieId,rating,timestamp
0,1,1,4.0,964982703
1,1,3,4.0,964981247
2,1,6,4.0,964982224
3,1,47,5.0,964983815
4,1,50,5.0,964982931


In [22]:
print(df_movies.shape)
print(df_ratings.shape)

(9742, 3)
(100836, 4)


In [23]:
df_movies.isnull().sum()
df_ratings.isnull().sum()

userId       0
movieId      0
rating       0
timestamp    0
dtype: int64

In [24]:
n_users = df_ratings['userId'].unique()
n_movies = df_movies['movieId'].unique()
print(n_users, n_movies)

[  1   2   3   4   5   6   7   8   9  10  11  12  13  14  15  16  17  18
  19  20  21  22  23  24  25  26  27  28  29  30  31  32  33  34  35  36
  37  38  39  40  41  42  43  44  45  46  47  48  49  50  51  52  53  54
  55  56  57  58  59  60  61  62  63  64  65  66  67  68  69  70  71  72
  73  74  75  76  77  78  79  80  81  82  83  84  85  86  87  88  89  90
  91  92  93  94  95  96  97  98  99 100 101 102 103 104 105 106 107 108
 109 110 111 112 113 114 115 116 117 118 119 120 121 122 123 124 125 126
 127 128 129 130 131 132 133 134 135 136 137 138 139 140 141 142 143 144
 145 146 147 148 149 150 151 152 153 154 155 156 157 158 159 160 161 162
 163 164 165 166 167 168 169 170 171 172 173 174 175 176 177 178 179 180
 181 182 183 184 185 186 187 188 189 190 191 192 193 194 195 196 197 198
 199 200 201 202 203 204 205 206 207 208 209 210 211 212 213 214 215 216
 217 218 219 220 221 222 223 224 225 226 227 228 229 230 231 232 233 234
 235 236 237 238 239 240 241 242 243 244 245 246 24

In [25]:
#Create User-Movie Matrix
user_movie_matrix = df_ratings.pivot(
    index = 'movieId',
    columns = 'userId',
    values = 'rating'
)

In [26]:
Y = user_movie_matrix.fillna(0).values

In [27]:
R = (user_movie_matrix.notnull()).astype(int).values

In [28]:
num_movies, num_users = Y.shape

print("\nMatrix shape:", Y.shape)


Matrix shape: (9724, 610)


In [29]:
# 5. NORMALIZE RATINGS (IMPORTANT)
# ================================
Y_mean = np.zeros((num_movies, 1))

for i in range(num_movies):
    idx = R[i, :] == 1
    if np.sum(idx) > 0:
        Y_mean[i] = np.mean(Y[i, idx])
    else:
        Y_mean[i] = 0

Y_norm = (Y - Y_mean) * R

In [30]:
# 6. INITIALIZE PARAMETERS
# ================================
num_features = 20  # tune this

X = np.random.randn(num_movies, num_features)
W = np.random.randn(num_users, num_features)
b = np.zeros((1, num_users))

In [31]:
# 7. COST FUNCTION
# ================================
def cofi_cost_func(X, W, b, Y, R, lambda_):
    nm, nu = Y.shape
    J = 0

    for j in range(nu):
        w = W[j,:]
        b_j = b[0,j]
        for i in range(nm):
            x = X[i,:]
            y = Y[i,j]
            r = R[i,j]
            J += 0.5 * np.square(r * (np.dot(w,x) + b_j - y))

    J += (lambda_/2) * (np.sum(np.square(W)) + np.sum(np.square(X)))
    return J

In [32]:
J = cofi_cost_func(X, W, b, Y, R, lambda_=1)
print(J)

1793035.2740388017


In [33]:
# 8. TRAINING (GRADIENT DESCENT)
# ================================
alpha = 0.001
lambda_ = 1
iterations = 100

for it in range(iterations):
    
    # predictions
    pred = X @ W.T + b
    
    # error only where rating exists
    error = (pred - Y_norm) * R
    
    # gradients
    X_grad = error @ W + lambda_ * X
    W_grad = error.T @ X + lambda_ * W
    b_grad = np.sum(error, axis=0, keepdims=True)
    
    # update
    X -= alpha * X_grad
    W -= alpha * W_grad
    b -= alpha * b_grad
    
    if it % 10 == 0:
        cost = cofi_cost_func(X, W, b, Y_norm, R, lambda_)
        print(f"Iteration {it}, Cost: {cost:.2f}")


Iteration 0, Cost: 618579.65
Iteration 10, Cost: 1133035.91


C:\Users\SUYASH\AppData\Local\Temp\ipykernel_5644\2889839352.py:10: RuntimeWarning: overflow encountered in matmul
  pred = X @ W.T + b
C:\Users\SUYASH\AppData\Local\Temp\ipykernel_5644\2889839352.py:10: RuntimeWarning: invalid value encountered in matmul
  pred = X @ W.T + b
C:\Users\SUYASH\AppData\Local\Temp\ipykernel_5644\2889839352.py:13: RuntimeWarning: invalid value encountered in multiply
  error = (pred - Y_norm) * R


Iteration 20, Cost: nan
Iteration 30, Cost: nan
Iteration 40, Cost: nan
Iteration 50, Cost: nan
Iteration 60, Cost: nan
Iteration 70, Cost: nan
Iteration 80, Cost: nan
Iteration 90, Cost: nan


In [34]:
# 9. PREDICTIONS
# ================================
pred = X @ W.T + b
pred = pred + Y_mean  # denormalize

In [35]:
# 10. RECOMMENDATION FUNCTION
# ================================
def recommend_movies(user_id, top_n=10):
    user_pred = pred[:, user_id]
    
    # sort
    movie_idx = np.argsort(user_pred)[::-1]
    
    movie_ids = user_movie_matrix.index[movie_idx]
    
    recommended = df_movies[df_movies['movieId'].isin(movie_ids[:top_n])]
    
    return recommended[['title']]

In [36]:
# 11. TEST RECOMMENDATION
# ================================
user_id = 0  # first user

print("\nTop Recommendations:")
print(recommend_movies(user_id, 10))


Top Recommendations:
                                                  title
9732                          Gintama: The Movie (2010)
9733  anohana: The Flower We Saw That Day - The Movi...
9734                                Silver Spoon (2014)
9735            Love Live! The School Idol Movie (2015)
9736           Jon Stewart Has Left the Building (2015)
9737          Black Butler: Book of the Atlantic (2017)
9738                       No Game No Life: Zero (2017)
9739                                       Flint (2017)
9740                Bungo Stray Dogs: Dead Apple (2018)
9741                Andrew Dice Clay: Dice Rules (1991)
